In [ ]:
# необходимые библиотеки для воспроизведения
#!pip install matplotlib==3.10.8 numpy==2.3.5 optuna==4.7.0 pandas==3.0.0 seaborn==0.13.2 scikit-learn==1.8.0 transformers==5.2.0 

In [ ]:
# необходимые библиотеки для воспроизведения
#pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu130

# Поиск фото по описанию

Для фотохостинга профессиональных фотографов «Со Смыслом» (“With Sense”) разработать функционал по поиску референсных (на основе тесктового описания) фотографий для фотографов.

Для демонтсрационной версии необходимо разработать модель, которая на основании векторного представления изображения и векторного представление текста на выходе выдаст число от 0 до 1 — и покажет, насколько текст и картинка подходят друг другу.

Также необходимо очистить данные от описаний и изображений детей (ребёнком считается любой человек, не достигший 16-ти лет)

Работа над проектом будет выполняться в несколько этапов:
- загрузка и изучение общей информации о данных;
- предобработка данных;
- исследовательский анализ данных;
- подготовка данных для модели МО;
- инициализации и обучению моделей МО с различными гиперпараметрами 

**Описание данных**
1. В файле `train_dataset.csv` находится информация, необходимая для обучения:
    - имя файла изображения;
    - идентификатор описания;
    - текст описания (для одной картинки может быть доступно до 5 описаний).
2. В папке `train_images` содержатся изображения для тренировки модели
3. В файле `CrowdAnnotations.tsv`  — данные по соответствию изображения и описания, полученные с помощью краудсорсинга:
    - имя файла изображения;
    - идентификатор описания;
    - доля людей, подтвердивших, что описание соответствует изображению;
    - количество человек, подтвердивших, что описание соответствует изображению;
    - количество человек, подтвердивших, что описание не соответствует изображению.
4. В файле `ExpertAnnotations.tsv`  — данные по соответствию изображения и описания, полученные в результате опроса экспертов^
    - имя файла изображения;
    - идентификатор описания;
    - оценки трёх экспертов.
5. В файле `test_queries.csv` находится информация, необходимая для тестирования:
    - идентификатор запроса;
    - текст запроса;
    - релевантное изображение.

In [ ]:
import os
import re
import time
import warnings

import copy
import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import Normalizer
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm import notebook, tqdm

from transformers import CLIPProcessor, CLIPModel
from scipy.stats import spearmanr
import torch.nn.functional as F

In [ ]:
# скрытие пердупреждений
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
RANDOM_STATE=42

In [ ]:
# путь к данным
PATH = 'data'

# путь к модели all-MiniLM-L6-v2
CLIP_PATH = '../../../models/CLIP'

# путь к папке с тренировочными изображениями
TRAIN_IMAGE_FOLDER = os.path.join(PATH, 'train_images')

## Загрузка данных

In [ ]:
train_dataset = pd.read_csv(os.path.join(PATH, 'train_dataset.csv'))
crowd_annotations = pd.read_csv(os.path.join(PATH, 'CrowdAnnotations.tsv'), sep='\t', header=None)
expert_annotations = pd.read_csv(os.path.join(PATH, 'ExpertAnnotations.tsv'), sep='\t', header=None)

In [ ]:
train_dataset.head()

In [ ]:
train_dataset.info()

In [ ]:
# количество уникальных изображеий и текстов
train_dataset[['image', 'query_id']].nunique()

In [ ]:
# вывод изображений на экран
image_folder = 'data/train_images'
sample_images = train_dataset.sample(16, random_state=RANDOM_STATE)

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('примеры изображений из тренировочного датасета', fontsize=14)

for idx, (_, row) in enumerate(sample_images.iterrows()):
    ax = axes[idx // 4, idx % 4]
    img = Image.open(os.path.join(image_folder, row['image']))
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
crowd_annotations.head()

In [ ]:
crowd_annotations.info()

In [ ]:
# переименование столбцов таблицы crowd_annotations
crowd_annotations.columns = ['image', 'query_id', 'assessment', 'yes_number', 'no_number']

In [ ]:
expert_annotations.head()

In [ ]:
expert_annotations.info()

In [ ]:
# переименование столбцов таблицы expert_annotations
expert_annotations.columns = ['image', 'query_id', 'expert_1', 'expert_2', 'expert_3']

<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

- На этапе изучения общей информации, в данных не обнаружено пропусков, возможно, присутствуют дубликаты (будут выявлены на следующем этапе).
- В тренировочном датасете 1000 уникальных изображений и 977 уникальных тестов.
- Таблице `crowd_annotations` присвоены наименования столбцов: image, query_id, assessment, yes_number, no_number.
- Таблице `expert_annotations` присвоены наименования столбцов: image, query_id, expert_1, expert_2, expert_3.

## Предобработка данных

На данном этапе будет проведена работа по: 
1. агрегации оценок соответствия текста и изображения в датафрейме expert_annotations и визуализация распределения количества оценок;
2. присоединию к `train_dataset` агрегированных оценок экспертов и доле людей, подтвердивших, что описание соответствует изображению:
     - присоединениие проведем на основании результатов проверки наличия пар наименование изображения - идентификатор описания из таблицы `train_dataset` в `expert_annotations` и`crowd_annotations`
3. выделению целевой переменной на основании присоединенных значений

### Агрегация оценок датафрейма expert_annotations

In [ ]:
# агрегация оценок на основе медианного значения оценок 3-х экспертов
expert_annotations['aggregated_assessment'] = expert_annotations[['expert_1', 'expert_2', 'expert_3']].median(axis=1).astype(int)
expert_annotations.head()

In [ ]:
# график распределения количества оценок
ax = expert_annotations.groupby('aggregated_assessment')['aggregated_assessment'].count() \
    .sort_values(ascending=False).plot(kind='bar',
                                       xlabel='assessments', 
                                       ylabel='number of assessment',
                                       figsize=(8, 5),
                                       width=0.5)

# настройка графика
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
ax.set_title(f'распределение количества сгруппированных данных для столбца: {'aggregated_assessment'}')
for bar_group in ax.containers:
     ax.bar_label(bar_group, label_type='edge')

По результатам визуализации распределения количества агренированных оценок экспертов установлено, что:
- в 57% случаев изображение и запрос совершенно не соответствуют друг другу (оценка 1);
- в 29% случаев запрос содержит элементы описания изображения, но в целом запрос тексту не соответствует (оценка 2);
- в 9% случаев запрос и текст соответствуют с точностью до некоторых деталей (оценка 3);
- в 5% случаев запрос и текст соответствуют полностью (оценка 4)

### Распределения крауд оценок

In [ ]:
# построим график распределения количества оценок
ax = crowd_annotations.groupby(crowd_annotations['assessment'].round(3))['assessment']\
    .count().sort_values(ascending=False).plot(kind='bar',
                                               xlabel='assessments',
                                               ylabel='number of assessmrnts',
                                               figsize=(14, 8))

# настройка графика
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_title('распределение количества уникальных оценок')
for bar_group in ax.containers:
    ax.bar_label(bar_group, label_type='edge')

По результатам визуализации распределения количества краудсорсинговых оценок установлено, что:
- в ~88 % случаев изображение и запрос совершенно не соответствуют друг другу (доля людей, подтвердивших, что описание соответствует изображению: **0**);
- в 6,2 % случаев изображение частично соответствует запросу (доля людей, подтвердивших, что описание соответствует изображению: **0,33**)
- в 2,8 % случаев изображение соответствует запросу с точностью до некоторых деталей (доля людей, подтвердивших, что описание соответствует изображению: **0,67**
- в 2,7 % случаев изображение и запрос полностью соответствуют друг другу (доля людей, подтвердивших, что описание соответствует изображению: **1**)

В данных краудсорсинговых оценок наблюдаестя более сильный дисбаланс, нежели в оценках эспертов

### Объединение таблиц

Проведем присоединение к таблице train_dataset агрегированных оценок экспертов (таблица expert_annotations) и доле людей, подтвердивших, что описание соответствует изображению (таблица crowd_annotations) на основании пары наименование изображения - идентификатор описания

\
Перед объединением проверим все ли пары (на основании которых будет проведено присоединение) присутствуют в датафреймах expert_annotations и crowd_annotations

In [ ]:
# множества пар из таблиц
train_pairs = set(zip(train_dataset['image'], train_dataset['query_id']))
expert_pairs = set(zip(expert_annotations['image'], expert_annotations['query_id']))
crowd_pairs = set(zip(crowd_annotations['image'], crowd_annotations['query_id']))

# проверка наличия пар: наименование изображения - идентификатор описания
# из train_dataset в expert_annotations
missing_in_expert = train_pairs - expert_pairs
print(f"количество пар из train_dataset, отсутствующих в expert_annotations: {len(missing_in_expert)}")

# проверка наличия пар наименование изображения - идентификатор описания
# из train_dataset в crowd_annotations
missing_in_crowd = train_pairs - crowd_pairs
print(f"количество пар из train_dataset, отсутствующих в crowd_annotations: {len(missing_in_crowd)}")


\
В таблице crowd_annotations отсутствует 3493 пары наименование изображения - идентификатор описания из таблицы train_dataset, поэтому проведем присоединение только агрегированных оценок экспертов (учет только оценок экспертов может улучшить качество обучения, т.к оценки экспертов более точные нежели краудсорсинговые)

In [ ]:
# присоединение к train_dataset агрегированных оценок экспертов
train_dataset_combined = train_dataset.merge(
    expert_annotations[['image', 'query_id', 'aggregated_assessment']],
    left_on=['image', 'query_id'],
    right_on=['image', 'query_id'],
    how='left'
)

train_dataset_combined.head()

In [ ]:
train_dataset_combined.info()

### Выделение целевой переменной

Приведем агрегированные значения оценок в диапазон от о до 1 путем минимаксной нормализации.

In [ ]:
from scipy.special import expit
def convert_to_probability(score):
    # логистическое преобразование
    return expit((score - 2.5) * 1.5) 

In [ ]:
# получение целевой переменной
train_dataset_combined['target'] = convert_to_probability(train_dataset_combined['aggregated_assessment'])

# удаления столбца с агрегированными оценками
train_dataset_combined = train_dataset_combined.drop('aggregated_assessment', axis=1)

In [ ]:
train_dataset_combined.head()

### Выводы

<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

1. Проведена работа по агрегации оценок экспертов соответствия текста и изображения в датафрейме `expert_annotations` и визуализация полученных результатов.
2. По результатам визуализации распределения количества агренированных оценок экспертов установлено, что:
    - в 57% случаев изображение и запрос совершенно не соответствуют друг другу (оценка 1);
    - в 29% случаев запрос содержит элементы описания изображения, но в целом запрос тексту не соответствует (оценка 2);
    - в 9% случаев запрос и текст соответствуют с точностью до некоторых деталей (оценка 3);
    - в 5% случаев запрос и текст соответствуют полностью (оценка 4).
    - выявленный дисбаланс, вероятно, повлияет на обучение модели в худшую сторону
3. По результатам визуализации распределения количества краудсорсинговых оценок установлено, что:
    - в ~88 % случаев изображение и запрос совершенно не соответствуют друг другу (доля людей, подтвердивших, что описание соответствует изображению: **0**);
    - в 6,2 % случаев изображение частично соответствует запросу (доля людей, подтвердивших, что описание соответствует изображению: **0,33**)
    - в 2,8 % случаев изображение соответствует запросу с точностью до некоторых деталей (доля людей, подтвердивших, что описание соответствует изображению: **0,67**
    - в 2,7 % случаев изображение и запрос полностью соответствуют друг другу (доля людей, подтвердивших, что описание соответствует изображению: **1**)
    - В данных краудсорсинговых оценок наблюдаестя более сильный дисбаланс, нежели в оценках эспертов
4. Установлено, что таблице `crowd_annotations` отсутствует 3493 пары *наименование изображения - идентификатор описания* из таблицы `train_dataset`. Основываясь на данном результате, а также визуализации распределения количества краудсорсинговых оценок, выделение целевой переменной будет проведено только исходя из агрегированных оценок экспертов путем присоединения к таблице `train_dataset` данных оценок (таблица `expert_annotations`) на комбинаций вышеуказанных пар (что может улучшить качество обучения, т.к оценки экспертов более точные нежели краудсорсинговые).
5. Проведено преобразование значений присоединенных оценок в диапазон от о до 1 путем логистического преобразования.

В результате получена таблица `train_dataset_combined` со следующими столбцами:
- имя файла изображения;
- идентификатор описания;
- текст описания;
- целевая переменная.

## Подготовка данных для модели МО

На данном этапе будет проведена работа по: 
1. очистке данных, в соответствии с юридическими ограничениями:
    - проведем очитку данных от лишних симоволов с использованием регулярных выражений;
    - составим список слов, которые могут попадать под юр. ограничения
    - исключим из таблицы `train_dataset_combined` от описаний и изображений детей (ребёнком считается любой человек, не достигший 16-ти лет).
2. векторизации тренировочных текстовых описаний и изображений:
    - инициализации процессора модели (в качестве модели используем CLIP);
    - преобразованию тестов в эмбеддинги
    - преобразованию изображений в эмбеддинги
3. разделению данных на тренировочную и валидационную выборки

### Очистка данных

In [ ]:

# функция очистки данных
def clean_text(text):
    '''
    функция для:
    - приведения текстов к нижнему регистру;
    - удаления цифр
    ''' 
    # удаление лишних пробелов в начале и конце
    text = text.strip()
    
    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s\.\,\!\?\(\)\']', ' ', text)

    # удаление множественных пробелов
    text = re.sub(r'\s+', ' ', text).strip()

    # удаление пробелов перед знаками препинания
    text = re.sub(r'\s+([\.\,\!\?\)])', r'\1', text)

    # удаление лишних пробелов в начале и конце
    text = text.strip()

    return text


In [ ]:
train_dataset_combined['cleaned_text'] = train_dataset_combined['query_text'].apply(clean_text)

# удаление стобца text
train_dataset_combined = train_dataset_combined.drop('query_text', axis=1)

train_dataset_combined.head()

In [ ]:
# список слов, попадающих под юр. ограничения
words = ['child', 'children', 'kid', 'kids', 'baby', 
         'little', 'young', 'tiny', 'schoolboy', 
         'schoolgirl', 'preschool', 'teen', 'teenager', 'youth', 
         'sandbox', 'toy', 'kindergarten', 'school', 'classroom']

In [ ]:
def contains_bad_words(text):
    '''
    функция проверки наличия слов из списка words
    в текстах
    '''
    if not isinstance(text, str):
        return False
    return any(word in text for word in words)

In [ ]:

# датафрейм, содержащий текст со словами из списка words
bad_comments = train_dataset_combined[train_dataset_combined['cleaned_text'].apply(contains_bad_words)]

# наименрвания изображений с детьми
bad_images = bad_comments['query_id'].str[:-2].unique()


In [ ]:
# фильтрация данных (исключаем изображения детей)
train_dataset_filtered = train_dataset_combined[~train_dataset_combined['query_id'].str[:-2].isin(bad_images)]

print(f"исходное количество строк: {len(train_dataset_combined)}")
print(f"количество уникальных изображений с детьми: {len(bad_images)}")
print(f"удалено строк с упоминанием детей: {len(bad_comments)}")
print(f"осталось строк: {len(train_dataset_filtered)}")

In [ ]:
train_images_path = os.path.join(PATH, 'train_images')

# вывод на экран удаленных изображений
bad_images_list = list(bad_images)[:16]

fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('примеры удаленных изображений из тренировочного датасета', fontsize=14)
axes = axes.flatten()

for idx, img_name in enumerate(bad_images_list):
    img = Image.open(os.path.join(train_images_path, img_name))
    axes[idx].imshow(img)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()


# Также выводим список имен для справки
print("\nнаименования, удаляемых изображений (первые 16):")
for i, img in enumerate(bad_images_list, 1):
    print(f"{i}. {img}")

In [ ]:
# проверка на дубликаты 
train_dataset_filtered.duplicated().sum()

In [ ]:
train_dataset_filtered.info()

### Векторизации текстов и изображений

In [ ]:
# инициализация процессора модели CLIP
processor = CLIPProcessor.from_pretrained(CLIP_PATH)

# инициализация модели
clip_model = CLIPModel.from_pretrained(CLIP_PATH).to(device)

In [ ]:
batch_size = 100

# тексты к изображениям
texts =  train_dataset_filtered['cleaned_text'].astype(str).tolist()

# наименования изображений
image_names = train_dataset_filtered['image']

In [ ]:
# преобразование текстов в эмбеддинги
text_embeddings = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i+batch_size]
    inputs = processor(
        text=batch_texts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=77
    ).to(device)

    with torch.no_grad():
        result = clip_model.get_text_features(**inputs)

        if isinstance(result, torch.Tensor):
            embeddings = result
        elif isinstance(result, tuple) and len(result) > 0:
            embeddings = result[0]  # Берем первый элемент
        elif hasattr(result, 'pooler_output'):
            embeddings = result.pooler_output
        elif hasattr(result, 'last_hidden_state'):
            embeddings = result.last_hidden_state.mean(dim=1)
        else:
            raise ValueError(f"Неизвестный формат: {type(result)}")
                
        # нормализация эмбеддингов
        embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)
        text_embeddings.append(embeddings.cpu().numpy())

text_embeddings = np.vstack(text_embeddings)
    

In [ ]:

# преобразование изображений в эмбеддинги
image_embeddings = []

for i in tqdm(range(0, len(image_names), batch_size)):
    batch_names = image_names[i:i+batch_size]

    for img_name in batch_names:
        image_path = os.path.join(TRAIN_IMAGE_FOLDER, img_name)
        image = Image.open(image_path).convert('RGB')
        inputs = processor(
            images=image, 
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            result = clip_model.get_image_features(**inputs)

            if isinstance(result, torch.Tensor):
                embeddings = result
            elif isinstance(result, tuple) and len(result) > 0:
                embeddings = result[0]  # Берем первый элемент
            elif hasattr(result, 'pooler_output'):
                embeddings = result.pooler_output
            elif hasattr(result, 'last_hidden_state'):
                embeddings = result.last_hidden_state.mean(dim=1)
            else:
                raise ValueError(f"Неизвестный формат: {type(result)}")
        
            embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)
            image_embeddings.append(embeddings.cpu().numpy())

image_embeddings = np.vstack(image_embeddings)


In [ ]:

# cохранение результатов вместе с текстовыми эмбеддингами
joblib.dump({
    'image' : train_dataset_filtered['image'].values,
    'query_id' : train_dataset_filtered['query_id'].values,
    'cleaned_text' : train_dataset_filtered['cleaned_text'].values,
    'text_embeddings': text_embeddings,
    'image_embeddings' : image_embeddings,
    'target': train_dataset_filtered['target'].values
}, 'text_image_embeddings.joblib', compress=0)


In [ ]:
print(f"количество пар текст-изображение: {len(image_embeddings)}")
print(f"размерность текстовых эмбеддингов: {text_embeddings.shape}")
print(f"размерность эмбеддингов изображений: {image_embeddings.shape}")

### Разделение данных на тренировочную и валидационную выборки

Разделение данных на тренировочную и валидационную выборки проведем так, чтобы уникальные изображения не попадали сразу в две выборками.\
Разделим данные в соотношении 90:10

In [ ]:
# загрузка данных
data = joblib.load('text_image_embeddings.joblib')

In [ ]:
# инициализация GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=RANDOM_STATE)


train_idx, val_idx = next(gss.split(
    X=data['text_embeddings'], 
    y=data['target'], 
    groups = data['image']
))

In [ ]:
X_train_text = data['text_embeddings'][train_idx]
X_train_image = data['image_embeddings'][train_idx]

X_val_text = data['text_embeddings'][val_idx]
X_val_image = data['image_embeddings'][val_idx]

y_train = data['target'][train_idx]
y_val = data['target'][val_idx]

In [ ]:
print(f"размер обучающей выборки с текстами: {X_train_text.shape}")
print(f"размер обучающей выборки с изображениями: {X_train_image.shape}")

print(f"размер валидационной выборки с текстами: {X_val_text.shape}")
print(f"размер валидационной выборки с изображениями: {X_val_image.shape}")

In [ ]:
y_train.shape, y_val.shape

### Выводы

<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

По результатам подготовки данных:
1. проведена очистка данных, в соответствии с юридическими ограничениями:
    - проведена очистка данных с использованием регулярных выражений
    - из таблицы `train_dataset_combined` исключено 1156 строк, содержащих описание и наименование изображений детей (ребёнком считается любой человек, не достигший 16-ти лет);
    - в результате получен очищенный датафрейм `train_dataset_filtered`, не содержащий дубликатов.
2. проведена векторизация текстов и изображений с использованием модели `CLIP`
4. По результатам векторизации текстов и изображений получены эмбеддинги с размерностью:
	- (4666, 512) для тестовых эмбеддингов (4666 - количество пар текст-изображение в `train_dataset_filtered`, 512 - размерность текстовых векторов);
	 - (4666, 512) для эмбеддингов изображений (4666 - количество пар текст-изображение в `train_dataset_filtered`, 512 – размер визуальных векторов)
5. проведено разделение данных на тренировочную и валидационную выборки в соотношении 90:10. Данные разделены так чтобы уникальные изображения не попадали сразу в две выборки.


## Обучение моделей МО

- На данном этапе будет проведена работа по инициализации и обучению моделей МО с различными гиперпараметрами.\
Используем модели: Ridge, Lasso, ElasticNet, LinearRegression, а также модель по определению косинусного сходства для  вычисления близости векторов текстов и изображений;
- Подбор наилучших гиперпараметров будем производить на основании значения метрики с помощью библиотеки optuna;
- В качесте основной метрики используем MSE.

In [ ]:
# скрытие сообщений (кроме ошибок)
optuna.logging.set_verbosity(optuna.logging.CRITICAL)

In [ ]:
# функция подбора гиперпараметров для линейных моделей
def objective_linear(trial, model_name, X_train_text, X_train_image, y_train,
                     X_val_text=None, X_val_image=None, y_val=None):
    '''
    функция подбора гиперпараметров для: 
    - моделей для моделей регрессии (Ridge, Lasso, ElasticNet, LinearRegression);
    - полносвязной нейросети
    на основании значения метрики MSE
    ''' 
    # объединение текстовых и визуальных эмбеддингов
    X_train = np.concatenate([X_train_text, X_train_image], axis=1)
    X_val = np.concatenate([X_val_text, X_val_image], axis=1)
    
    # общие параметры
    common_params = {
        'random_state': RANDOM_STATE,
        'max_iter': 2000
    }
    
    # гиперпараметры для Ridge
    if model_name == 'ridge':
        params = {
            'alpha': trial.suggest_float('alpha', 1e-3, 5.0, log=True),
            'solver': trial.suggest_categorical('solver', ['auto', 'svd', 'cholesky', 'lsqr', 'sag'])
        }
        params.update(common_params)
        model = Ridge(**params)

    # гиперпараметры для Lasso
    elif model_name == 'lasso':
        params = {
            'alpha': trial.suggest_float('alpha', 1e-4, 5.0, log=True), 
            'selection': trial.suggest_categorical('selection', ['cyclic', 'random'])
        }
        params.update(common_params)
        model = Lasso(**params)

    # гиперпараметры для ElasticNet
    elif model_name == 'elastic':
        params = {
            'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),  
            'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),  
            'selection': trial.suggest_categorical('selection', ['cyclic', 'random'])
        }
        params.update(common_params)
        model = ElasticNet(**params)
    
    # игициализация LinearRegression
    elif model_name == 'lr':
        model = LinearRegression()

    # обучение моделей
    model.fit(X_train, y_train)
    
    # получение предсказаний
    y_pred = model.predict(X_val)
    
    # расчет метрик
    mse = mean_squared_error(y_val, y_pred)
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    # среднее значение метрик
    metrics = {
        'mse': round(mse, 3),
        'mae': round(mae, 3),
        'r2': round(r2, 3)
    }
    # сохранение метрик
    for name, value in metrics.items():
        trial.set_user_attr(name, value)

    return metrics['mse']

### Обучение Ridge

In [ ]:
start = time.time()

study_ridge = optuna.create_study(direction='minimize')
study_ridge.optimize(
    lambda trial: objective_linear(
        trial, 
        'ridge', 
        X_train_text, X_train_image, y_train,
        X_val_text, X_val_image, y_val
    ), 
    n_trials=30, show_progress_bar=True
)

print(f"время подбора гиперпараметров: {time.time() - start:.2f}\n")
print('лучшие параметры', study_ridge.best_params)
print('\nлучшее значение MSE при кроссвалидации:',
      study_ridge.best_value)

In [ ]:
# лучшие параметры модели
best_trial_ridge = study_ridge.best_trial

### Обучение Lasso

In [ ]:
start = time.time()

study_lasso = optuna.create_study(direction='minimize')
study_lasso.optimize(
    lambda trial: objective_linear(
        trial, 
        'lasso',
        X_train_text, X_train_image, y_train,
        X_val_text, X_val_image, y_val
    ),
    n_trials=30, show_progress_bar=True
)

print(f"время подбора гиперпараметров: {time.time() - start:.2f}\n")
print('лучшие параметры', study_lasso.best_params)
print('\nлучшее значение MSE при кроссвалидации:',
      study_lasso.best_value)

In [ ]:
# лучшие параметры модели
best_trial_lasso = study_lasso.best_trial

### Обучение ElasticNet

In [ ]:
start = time.time()

study_elastic = optuna.create_study(direction='minimize')
study_elastic.optimize(
    lambda trial: objective_linear(
        trial, 
        'elastic', 
        X_train_text, X_train_image, y_train,
        X_val_text, X_val_image, y_val
    ),
    n_trials=30, show_progress_bar=True
)

print(f"время подбора гиперпараметров: {time.time() - start:.2f}\n")
print('лучшие параметры', study_elastic.best_params)
print('\nлучшее значение MSE при кроссвалидации:',
      study_elastic.best_value)

In [ ]:
# лучшие параметры модели
best_trial_elastic = study_elastic.best_trial

### Обучение LinearRegrission

In [ ]:
start = time.time()

study_lr = optuna.create_study(direction='minimize')
study_lr.optimize(
    lambda trial: objective_linear(
        trial, 
        'lr', 
        X_train_text, X_train_image, y_train,
        X_val_text, X_val_image, y_val
    ),
    n_trials=30, show_progress_bar=True
)

print(f"время подбора гиперпараметров: {time.time() - start:.2f}\n")
print('лучшие параметры', study_lr.best_params)
print('\nлучшее значение MSE при кроссвалидации:',
      study_lr.best_value)

In [ ]:
# лучшие параметры модели
best_trial_lr = study_lr.best_trial

### Инициализация и обучение нейросети

In [ ]:
class CosineModel(nn.Module):
    '''
    класс инициализации модели для вычисления близости текстов
    и изображений через косинусное сходство

    используемые параметры:
    text_dim - размерность текстовых эмбеддингов;
    image_dim - размерность эмбеддингов изображений;
    projection_dim - размерность общего пространства;
    dropout - регуляризация;
    use_projection (True или False) - использовать ли проекционные слои
    для приведения текстов и изображений к общему пространству
    '''
    def __init__(self, text_dim, image_dim,
                 projection_dim, dropout, 
                 use_projection):
        super().__init__()

        self.use_projection = use_projection
        # слои для приведения текстов и изображений к общему пространству
        if use_projection:
            self.text_projection = nn.Sequential(
                nn.Linear(text_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
            
            self.image_projection = nn.Sequential(
                nn.Linear(image_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            )

            self.projection_dim = projection_dim
            
        else:
            self.projection_dim = text_dim

        # регрессионный слой для доп. преобразования распределения
        # выходных значений
        self.regression_head = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 1)
        )

        self._init_weights()
        
    # функция инициализации весов
    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, text_embeddings, image_embeddings):
        if self.use_projection:
            text_projection = self.text_projection(text_embeddings)
            image_projection = self.image_projection(image_embeddings)
        else:
            text_projection = text_embeddings
            image_projection = image_embeddings

        text_projection = F.normalize(text_projection, p=2, dim=1)
        image_projection = F.normalize(image_projection, p=2, dim=1)

        cosine_similarity = torch.sum(text_projection * image_projection, dim=1, keepdim=True)

        modified = self.regression_head(cosine_similarity)

        output = torch.sigmoid(modified )

        return output

In [ ]:
def training_cosine_model(model, X_train_text, X_train_image, y_train, 
             X_val_text, X_val_image, y_val, params, device):
    '''функция обучения модели косинусного сходства с early stopping'''
    
    # преобразование данных в тензоры 
    X_train_text = torch.FloatTensor(X_train_text)
    X_train_image = torch.FloatTensor(X_train_image)
    y_train = torch.FloatTensor(y_train).view(-1, 1)

    X_val_text = torch.FloatTensor(X_val_text)
    X_val_image = torch.FloatTensor(X_val_image)
    y_val = torch.FloatTensor(y_val).view(-1, 1)
    
    # инициализация датасетов и лоадеров
    train_dataset = TensorDataset(X_train_text, X_train_image, y_train)
    train_loader = DataLoader(
        train_dataset,
        batch_size=params['batch_size'],
        shuffle=True,
        pin_memory=True if device.type == 'cuda' else False
    )
    
    val_dataset = TensorDataset(X_val_text, X_val_image, y_val)
    val_loader = DataLoader(
        val_dataset,
        batch_size=params['batch_size'],
        shuffle=False,
        pin_memory=True if device.type == 'cuda' else False
    )
    
    # обработка оптимизаторов
    if params['optimizer'] == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), 
                                     lr=params['learning_rate'],
                                     weight_decay=params['weight_decay'])
    elif params['optimizer'] == 'adamw':
        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=params['learning_rate'],
                                      weight_decay=params['weight_decay'])
    else:
        optimizer = torch.optim.SGD(model.parameters(),
                                    lr=params['learning_rate'],
                                    momentum=0.9,
                                    weight_decay=params['weight_decay'])
    
    # scheduler для уменьшения learning rate
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )
    
    # функция потерь
    criterion = nn.MSELoss()
    
    # переменные для отслеживания лучшего результата
    best_val_loss = float('inf')
    best_model_state = None
    best_metrics = None
    patience_counter = 0
    early_stop_patience = 20 
    
    # обучение нейросети
    for epoch in range(params['epochs']):
        # обучение
        model.train()
        epoch_loss = 0.0
        
        for batch_text, batch_image, batch_y in train_loader:
            batch_text = batch_text.to(device)
            batch_image = batch_image.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            preds = model(batch_text, batch_image)
            loss_value = criterion(preds, batch_y)
            loss_value.backward()
            
            # градиентное клиппинг для стабильности
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            epoch_loss += loss_value.item()
        
        # валидация (каждую эпоху)
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_targets = []
        
        with torch.no_grad():
            for batch_text, batch_image, batch_y in val_loader:
                batch_text = batch_text.to(device)
                batch_image = batch_image.to(device)
                batch_y = batch_y.to(device)
                
                val_preds = model(batch_text, batch_image)
                val_loss += criterion(val_preds, batch_y).item()
                
                all_preds.append(val_preds.cpu())
                all_targets.append(batch_y.cpu())
                
        all_preds = torch.cat(all_preds).numpy().flatten()
        all_targets = torch.cat(all_targets).numpy().flatten()
        
        # обновление scheduler
        scheduler.step(val_loss)
        
        # сохранение лучшей модели
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            
            # вычисление метрик для лучшей модели
            val_preds_np = val_preds.cpu().numpy().flatten()
            y_val_np = y_val.cpu().numpy().flatten()
                
            best_metrics = {
                'loss': val_loss,
                'mse': mean_squared_error(all_targets, all_preds),
                'mae': mean_absolute_error(all_targets, all_preds),
                'r2': r2_score(all_targets, all_preds),
                'spearman': spearmanr(all_targets, all_preds)[0]
            }
            
            patience_counter = 0
        else:
            patience_counter += 1
            
        # ранняя остановка
        if patience_counter >= early_stop_patience:
            print(f'ранняя остановка на эпохе {epoch}')
            break
    
    # загрузка лучшей модели
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    # возвращаем модель метрики лучшей модели
    return model, best_metrics

In [ ]:
# переменные для хранения лучшей модели
best_trained_model = None
best_trial_params = None
best_metrics_global = None

# функция подбора гиперпараметров для линейных моделей
def objective_cosine(trial, X_train_text, X_train_image, y_train, 
                     X_val_text, X_val_image, y_val, device):
    '''
    функция подбора гиперпараметров для
    модели косинусного сходства
    на основании значения метрики MSE
    '''
    try:
        global best_trained_model, best_trial_params
        
        params = {
            # параметры архитектуры проекции
            'projection_dim': trial.suggest_int('projection_dim', 64, 512, log=True),
            'use_projection': trial.suggest_categorical('use_projection', [True, False]),
            'dropout': trial.suggest_float('dropout', 0.0, 0.5),  
            
            # параметры обучения
            'batch_size': trial.suggest_categorical('batch_size', [256, 512, 1024]),
            'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('optimizer', ['adam', 'adamw', 'sgd']),
            'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
            'epochs': trial.suggest_int('epochs', 50, 300)
        }
    
        model = CosineModel(
            text_dim=X_train_text.shape[1],
            image_dim=X_train_image.shape[1],
            projection_dim=params['projection_dim'],
            dropout=params['dropout'],
            use_projection=params['use_projection']
        ).to(device)
            
    
        # обучение с валидацией
        model, metrics = training_cosine_model(
            model, X_train_text, X_train_image, y_train,
            X_val_text,X_val_image, y_val,
            params, device
         )
    
        trial.set_user_attr('mae', metrics['mae'])
        trial.set_user_attr('r2', metrics['r2'])
        trial.set_user_attr('spearman', metrics['spearman'])
    
        current_best_value = float('inf')
        if len(trial.study.trials) > 0 and trial.study.best_trial is not None:
            current_best_value = trial.study.best_value
                
        # сохранение полной информации о модели
        if best_trained_model is None or metrics['mse'] < current_best_value:
            best_trained_model = copy.deepcopy(model)
            best_trial_params = params.copy()
            best_metrics_global = metrics.copy()
            torch.save({
                'model_state_dict': best_trained_model.state_dict(),
                'model_params': {
                    'text_dim': X_train_text.shape[1],
                    'image_dim': X_train_image.shape[1],
                    'projection_dim': params['projection_dim'],
                    'dropout': params['dropout'],
                    'use_projection': params['use_projection']
                },
                'training_params': params,
                'val_metrics': metrics
            }, 'best_cosine_model.pth')
            print(f"модель сохранена с mse: {metrics['mse']:.4f}")
    
        return metrics['mse']

    except Exception as e:
        print(f"Ошибка в trial {trial.number}: {e}")
        return float('inf')

In [ ]:
# запуск обучения нейросети
start = time.time()

study_nn = optuna.create_study(direction='minimize',
                               sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                               pruner=optuna.pruners.MedianPruner(
                                   n_startup_trials=5,
                                   n_warmup_steps=30,
                                   interval_steps=1
                               ))
study_nn.optimize(
    lambda trial: objective_cosine(trial, X_train_text, X_train_image, y_train, 
                                   X_val_text, X_val_image, y_val, device),
    n_trials=30, show_progress_bar=True
)

print(f"\nвремя подбора гиперпараметров: {time.time() - start:.2f}\n")
print('лучшие параметры', study_nn.best_params)
print('\nлучшее значение MSE:',
      study_nn.best_value)

In [ ]:
# лучшие параметры модели
best_trial_nn = study_nn.best_trial

### Таблица с результатами обучения моделей

In [ ]:
results = pd.DataFrame(
    {
        'модель' : ['Ridge', 'Lasso', 'ElasticNet', 'LinearRegrission', 'FeedForwardNN'],
        'mse' : [
            study_ridge.best_value, 
            study_lasso.best_value, 
            study_elastic.best_value,
            study_lr.best_value,
            study_nn.best_value
        ],
        'mae' : [
            best_trial_ridge.user_attrs['mae'],
            best_trial_lasso.user_attrs['mae'],
            best_trial_elastic.user_attrs['mae'],
            best_trial_lr.user_attrs['mae'],
            best_trial_nn.user_attrs['mae']
        ],
        'r2' : [
            best_trial_ridge.user_attrs['r2'],
            best_trial_lasso.user_attrs['r2'],
            best_trial_elastic.user_attrs['r2'],
            best_trial_lr.user_attrs['r2'],
            best_trial_nn.user_attrs['r2']
        ]
    }
)

In [ ]:
# округление всех числовых колонок
numeric_cols = ['mse', 'mae', 'r2']
results[numeric_cols] = results[numeric_cols].round(3)

results.sort_values('mse')

### Выводы

<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

1. Написана функция `objective` по подбору гиперпараметров для моделей.
    - подбор лучших гиперпараметров проведен на основании значения метрики MSE при кроссвалидации с помощью библиотеки optuna;
    - в качестве моделей выбраны: Ridge, Lasso, ElasticNet, LinearRegression и модель на основе косинусного сходстваь.
2. Для обучения нейросети написан класс `CosineModel` и функция `training_cosine_model`.
3. По результатам обучения моделей лушее значение метрики MSE при кроссвалидации показала модель на основе косинусного сходства (0.022)

## Тест лучшей модели

На данном этапе будет проведена работа по:
- инициализации нейросети с подобранными параметрами;
- инициализации тестового датасета;
- векторизации тестовых изображений с использованием модели CLIP;
- написанию функции, принимающей на вход текстовое описание и возвращающей максимально приближенное к описанию изображение с использованием обученной модели

### Инициализация модели

Проведем инициализацию нейросети с подобранными параметрами 

In [ ]:
# загрузка обученной нейросети
checkpoint = torch.load('best_cosine_model.pth', map_location=device, weights_only=False)

# параметры модели
model_params = checkpoint['model_params']
training_params = checkpoint['training_params']
val_metrics = checkpoint['val_metrics']

dropout_value = model_params.get('dropout', 0.0)
if dropout_value is None:
    dropout_value = 0.0
    
# инициализация модели
final_model = CosineModel(
    text_dim=model_params['text_dim'], 
    image_dim=model_params['image_dim'],
    projection_dim=model_params['projection_dim'],
    dropout=model_params['dropout'],
    use_projection=model_params['use_projection']
)

# загрузка весов модели
final_model.load_state_dict(checkpoint['model_state_dict'])

# перемещение модели на GPU
final_model = final_model.to(device)

# переключение в режим оценки
final_model.eval()

### Инициализация CLIP

In [ ]:
# инициализация процессора и модели
processor = CLIPProcessor.from_pretrained(CLIP_PATH)

# Инициализация модели и перемещение на GPU
clip_model = CLIPModel.from_pretrained(CLIP_PATH).to(device)
clip_model.eval() 

### Инициализация тестового датасета

In [ ]:
# инициализация тестового 
test_dataset = pd.read_csv(os.path.join(PATH, 'test_queries.csv'), sep='|')
test_dataset = test_dataset.drop('Unnamed: 0', axis=1)

test_images = pd.read_csv(os.path.join(PATH, 'test_images.csv'))

In [ ]:
test_dataset.head()

In [ ]:
test_dataset.info()

In [ ]:
test_dataset.nunique()

In [ ]:
test_images.head()

In [ ]:
test_images.info()

\
В тестовом датасете 100 уникальных изображений и 500 уникальных текстов

### Векторизация тестовых изображений

Перобразуем тестовые изображения в эмбеддинги и сформируем словарь, где ключ - наименование изображения, значение - эмбеддинг изображения.\
Векторизацию проведем с использованием CLIP, инициализированной ранее

In [ ]:
batch_size = 100

# путь к папке с изображениями
test_image_folder = os.path.join(PATH, 'test_images')

# наименования изображений
test_image_names = test_images['image']

In [ ]:
# преобразование изображений в эмбеддинги
embeddings_list = []
image_names_list = []

for i in tqdm(range(0, len(test_image_names), batch_size)):
    batch_names = test_image_names[i:i+batch_size].tolist()

    batch_images = []
    for img_name in batch_names:
        image_path = os.path.join(test_image_folder, img_name)
        image = Image.open(image_path).convert('RGB')
        batch_images.append(image)
        
    inputs = processor(
        images=batch_images, 
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        result = clip_model.get_image_features(**inputs)

        if isinstance(result, torch.Tensor):
            embeddings = result
        elif isinstance(result, tuple) and len(result) > 0:
            embeddings = result[0] 
        elif hasattr(result, 'pooler_output'):
            embeddings = result.pooler_output
        elif hasattr(result, 'last_hidden_state'):
            embeddings = result.last_hidden_state.mean(dim=1)
        else:
            raise ValueError(f"Неизвестный формат: {type(result)}")
    
        embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)
        embeddings_np = embeddings.cpu().numpy().astype(np.float32)

        for img_name, embedding in zip(batch_names, embeddings_np):
            image_names_list.append(img_name)
            embeddings_list.append(embedding)

# датафрейм наименований и эмбеддингов изображений
test_image_data = pd.DataFrame({
    'image': image_names_list,
    'embedding': embeddings_list
})

### Функция тестирования модели

Напишем функцию, которая:
- принимает на вхох текстовое описание, если текст содержит слова для вывода изображений с детьми, то на экране отобразится дисклеймер (This image is unavailable in your country in compliance with local laws)
- производит векторизацию полученного текста с использованием CLIP
- с помощью обученной модели вычисляет сходство полученного вектора текста с вектором изображения из тестовой выборки
- на выходе на экран выводится изображение максимально приближенное к описанию

In [ ]:
# список слов, попадающих под юр. ограничения
words = ['child', 'children', 'kid', 'kids', 'baby', 
         'little', 'small', 'young', 'tiny', 'schoolboy', 
         'schoolgirl', 'preschool', 'teen', 'teenager', 'youth', 
         'sandbox', 'toy', 'kindergarten', 'school', 'classroom',
         'boy', 'boys', 'girl', 'girls']

In [ ]:
def test_model(text):
    '''
    функция вывода на экран изображения из тестовой выборки
    на основании текстового описания
    text - описание для поика референсных изображений
    '''
    # подготовка текста и использованием функции clean_text
    text = clean_text(text)

    # определение слов, попадающих под юр. ограничения
    if any(word in text for word in words):
        print('\033[1mThis image is unavailable in your country in compliance with local laws\033[0m')
        return None, None

    # векторизация текста
    inputs = processor(
        text=text, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=77
    ).to(device)
    
    with torch.no_grad():
        result = clip_model.get_text_features(**inputs)
    
        if isinstance(result, torch.Tensor):
            embeddings = result
        elif isinstance(result, tuple) and len(result) > 0:
            embeddings = result[0] 
        elif hasattr(result, 'pooler_output'):
            embeddings = result.pooler_output
        elif hasattr(result, 'last_hidden_state'):
            embeddings = result.last_hidden_state.mean(dim=1)
        else:
            raise ValueError(f"Неизвестный формат: {type(result)}")
                
        # нормализация эмбеддингов
        embeddings = embeddings / embeddings.norm(dim=1, keepdim=True)
        text_embedding = embeddings.cpu().numpy()

    # преобразование эмбеддинга текста в тензор
    text_tensor = torch.FloatTensor(text_embedding).to(device)

    scores = [] 
    for idx, row in test_image_data.iterrows():
        image_name = row['image']
        image_embedding = row['embedding']

        # преобразование эмбеддинга изображения в тензор
        image_tensor = torch.FloatTensor(image_embedding.reshape(1, -1)).to(device)

        with torch.no_grad():           
            predict = final_model(text_tensor, image_tensor)
            score = predict.cpu().numpy()[0][0]
            
        scores.append((image_name, score))

    # сортировка предсказаний модели по убыванию
    scores.sort(key=lambda x: x[1], reverse=True)

    # вывод лучшего изображения
    best_image = scores[0][0]
    best_score = scores[0][1]

    print(f"лучшее предсказанное изображение: {best_image}")
    print(f"оценка сходства: {best_score:.4f}")
    
    image_path = os.path.join(test_image_folder, best_image)
    image = Image.open(image_path)
    plt.imshow(image)
    plt.axis('off')
    plt.show()

    return best_image, best_score

In [ ]:
# количество случайных текстов из тетового датасета
n_texts = 10

random_samples = test_dataset.sample(n=n_texts, random_state=RANDOM_STATE)
for idx, row in random_samples.iterrows():
    print(f'строка №{idx+1}')
    print(f'идентификатор описания: {row['query_id']}')
    print(f'текст: \033[1m{row['query_text']}\033[0m')
    print(f"изображение соответствующее тексту: {row['image']}")

    best_image, best_score = test_model(row['query_text'])
    if best_image is not None and best_score is not None:
        print(f'результат: {best_image} (score: {best_score:.4f})\n')
    else:
        print('\n')

### Выводы

<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

По результатам теста лучшей модели:
- проведена инициализация лучшей модели (полносвязная нейросеть)
- инициализирован тестовый датасет (таблица `test_dataset` и `test_images `) 
- в тестовых данных 100 уникальных изображений и 500 уникальных текстов
- проведена векторизация тестовых изображений;
- написана функция `test_model` вывода на экран изображения из тестовой выборки на основании текстового описания (для тестирования лучшей модели)
- получаемые, при тестировании модели, изображения иногда не соответствуют текстовому запросу, что свидетельствует о низкой предсказательной способности модели. 
- Для создания сервиса поиска фотографий по текстовому описанию необходимо существенно расширить обучающую выборку и качество разметки, чтобы для разных изображений не было одного и того же текстового описания.


## Выводы

<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

**Поставленная задача**\
Представить концепт проекта по поиску референсных изображений на основании тестового описания.

**В процессе работы над проектом:**
1. Была проведена работа по изучению данных:
	- в данных не обнаружено пропусков
	- в тренировочном датасете 1000 уникальных изображений и 977 уникальных тестов


<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

2. Проведена предобработка данных:
- Проведена работа по агрегации оценок экспертов соответствия текста и изображения в датафрейме `expert_annotations` и визуализация полученных результатов
-  По результатам визуализации распределения количества агренированных оценок экспертов установлено, что:
    -  в 57% случаев изображение и запрос совершенно не соответствуют друг другу (оценка 1);
    - в 29% случаев запрос содержит элементы описания изображения, но в целом запрос тексту не соответствует (оценка 2);
    - в 9% случаев запрос и текст соответствуют с точностью до некоторых деталей (оценка 3);
    - в 5% случаев запрос и текст соответствуют полностью (оценка 4).
    - выявленный дисбаланс, вероятно, повлияет на обучение модели в худшую сторону.
- По результатам визуализации распределения количества краудсорсинговых оценок установлено, что:
    - в ~88 % случаев изображение и запрос совершенно не соответствуют друг другу (доля людей, подтвердивших, что описание соответствует изображению: **0**);
    - в 6,2 % случаев изображение частично соответствует запросу (доля людей, подтвердивших, что описание соответствует изображению: **0,33**)
    - в 2,8 % случаев изображение соответствует запросу с точностью до некоторых деталей (доля людей, подтвердивших, что описание соответствует изображению: **0,67**
    - в 2,7 % случаев изображение и запрос полностью соответствуют друг другу (доля людей, подтвердивших, что описание соответствует изображению: **1**)
    - В данных краудсорсинговых оценок наблюдаестя более сильный дисбаланс, нежели в оценках эспертов
- Установлено, что таблице `crowd_annotations` отсутствует 3493 пары *наименование изображения - идентификатор описания* из таблицы `train_dataset`. Основываясь на данном результате, а также визуализации распределения количества краудсорсинговых оценок, выделение целевой переменной будет проведено только исходя из агрегированных оценок экспертов путем присоединения к таблице `train_dataset` данных оценок (таблица `expert_annotations`) на комбинаций вышеуказанных пар (что может улучшить качество обучения, т.к оценки экспертов более точные нежели краудсорсинговые).
- Проведено преобразование значений присоединенных оценок в диапазон от о до 1 путем минимаксной нормализации.
- В результате получена таблица `train_dataset_combined` со следующими столбцами:
- имя файла изображения;
- идентификатор описания;
- текст описания;
- целевая переменная.


<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

3. Проведена подготовка данных для модели МО по результатам которой:
- проведена очистка данных, в соответствии с юридическими ограничениями:
	- проведена очистка данных с использованием регулярных выражений
    - из таблицы `train_dataset_combined` исключено 1806 строк, содержащих описание и наименование изображений детей (ребёнком считается любой человек, не достигший 16-ти лет);
	- в результате получен очищенный датафрейм `train_dataset_filtered`, не содержащий дубликатов.
- проведена векторизация текстов и изображений с использованием модели CLIP
- по результатам векторизации текстов и изображений получены эмбеддинги с размерностью:
	- (4666, 512) для тестовых эмбеддингов (4666 - количество пар текст-изображение в `train_dataset_filtered`, 512 - размерность текстовых векторов);
	- (4666, 512) для эмбеддингов изображений (4666 - количество пар текст-изображение в `train_dataset_filtered`, 512 – размер визуальных векторов).
- проведено разделение данных на тренировочную и валидационную выборки в соотношении 90:10. Данные разделены так чтобы уникальные изображения не попадали сразу в две выборки.


<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

4. Проведено обучение моделей МО с различными гиперпараметрами:
- написана функция `objective` по подбору гиперпараметров для моделей:
	- подбор лучших гиперпараметров проведен на основании значения метрики MSE при кроссвалидации с помощью библиотеки optuna;
	- в качестве моделей выбраны: Ridge, Lasso, ElasticNet, LinearRegression и модель на основе косинусного сходства
- для обучения нейросети написан класс `CosineModel` и функция `training_cosine_model`
- по результатам обучения моделей лушее значение метрики MSE при кроссвалидации показала полносвязная нейросеть (0.022).


<div class="alert alert-info" style="border-radius: 10px; box-shadow: 2px 2px 2px; border: 1px solid; padding: 10px ">

5. Проведено тестирование лучшей модели:
- проведена инициализация лучшей модели (полносвязная нейросеть)
- инициализирован тестовый датасет (таблица `test_dataset` и `test_images `) 
- в тестовых данных 100 уникальных изображений и 500 уникальных текстов
- проведена векторизация тестовых изображений;
- написана функция `test_model` вывода на экран изображения из тестовой выборки на основании текстового описания (для тестирования лучшей модели)
- получаемые, при тестировании модели, изображения иногда не соответствуют текстовому запросу, что свидетельствует о низкой предсказательной способности модели. 
- Для создания сервиса поиска фотографий по текстовому описанию необходимо существенно расширить обучающую выборку и качество разметки, чтобы для разных изображений не было одного и того же текстового описания. Возможно для векторизации изображений необходимо использовать другие модели
